# HotpotQA · build the questions (local)

Writes `questions.pkl`: HotpotQA 2-hop questions, each with its 2 supporting paragraphs and 8 distractor
paragraphs turned into facts by Llama-3.1-70B, in a fixed nested order. Every notebook in this folder reads it.
The steps and filters are described at the top of `common/build.py`. Cost: about $1 of 70B calls.

**Run once.** If `questions.pkl` exists, this notebook only shows it; set `REBUILD = True` to build a new set (then
move `results/` away: answers to an old question set are never mixed with a new one).

In [1]:
import os, pickle, sys
sys.path.insert(0, "../common")

DATASET = "HotpotQA"
N_QUESTIONS = 200
K_MAX = 8            # distractor paragraphs per question (k = 0, 4, 8)
REBUILD = False

In [2]:
if os.path.exists("questions.pkl") and not REBUILD:
    print("questions.pkl already exists: set REBUILD = True to build a new set")
else:
    from build import build
    build(DATASET, N_QUESTIONS, K_MAX)

/Users/mina/miniforge3/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1. validation: 5918 2-hop questions, 5401 fresh with at least 8 distractors


Extracting facts: 100%|██████████| 360/360 [02:09<00:00,  2.79it/s]


   validation: {'screened': 400, 'unknown': 95, 'answerable': 60, 'selected': 36, 'complete': 36} → 36 questions so far


Extracting facts: 100%|██████████| 1740/1740 [09:37<00:00,  3.01it/s]

   validation: {'screened': 800, 'unknown': 311, 'answerable': 250, 'selected': 210, 'complete': 200} → 200 questions so far
Saved 200 questions to questions.pkl {'validation': 200}. API: {'model': 'meta-llama/Meta-Llama-3.1-70B-Instruct-Turbo', 'requests': 3760, 'failed': 0, 'input_tokens': 668530, 'output_tokens': 406755, 'cost_usd': 0.4301}


In [3]:
from collections import Counter
questions = pickle.load(open("questions.pkl", "rb"))
print(f"{len(questions)} questions · sources {dict(Counter(q['source_split'] for q in questions))} · "
      f"{len({q['answer'] for q in questions})} distinct answers")
for q in questions[:5]:
    supporting = sum(len(p["facts"]) for p in q["layout"] if p["is_supporting"])
    distractor = sum(len(p["facts"]) for p in q["layout"] if not p["is_supporting"])
    print(f"- {q['question']}  →  {q['answer']}   ({supporting} supporting facts, {distractor} in distractors)")

200 questions · sources {'validation': 200} · 200 distinct answers
- Lesley Foxcroft worked with MDF, Card, and which material that is produced by pressing together moist fibres?  →  Paper   (11 supporting facts, 109 in distractors)
- What is the birthday of the actress who was the Duchess in "The Revengers Tragedy"?  →  23 November 1946   (17 supporting facts, 60 in distractors)
- What word or phrase is found in both the history of Belgium and cockfighting?  →  cockpit   (18 supporting facts, 66 in distractors)
- What Labor member of the House of Representatives in the Australian Parliament was a member of the Australian Labor Party Australian Young Labor?  →  Tony Burke   (25 supporting facts, 135 in distractors)
- In what county was Duffy Jackson born?  →  Nassau County   (13 supporting facts, 84 in distractors)
